<a href="https://colab.research.google.com/github/BurhanAmin/SentimentAnalyser/blob/main/SentimentAnlysis(202)%5BFinal%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sentiment analysis — RoBERTa fine-tune

Fine-tunes `cardiffnlp/twitter-roberta-base-sentiment` on the Twitter Entity Sentiment
dataset into a three-class classifier: **0 = negative, 1 = neutral, 2 = positive**.

Runs on **Google Colab (GPU)** or locally on Apple silicon (MPS) / CPU — the device, mixed
precision and dataloader workers are all picked automatically.

**On Colab: set the runtime to a GPU first** — *Runtime → Change runtime type → T4 GPU* —
then run every cell in order. Section 2 handles getting the two CSVs into the session.

The trained model is written to `app/model/`, which is where the Flask app in `app/` looks
for it. Section 12 zips it for download.

## 1. Setup

The environment is already installed; uncomment to install elsewhere.

In [ ]:
# Colab ships transformers 4.x; this notebook uses the v5 API (`eval_strategy`,
# `processing_class`, no `logging_dir`). Installing is a no-op if you are already on v5.
# Restart the runtime if Colab asks you to, then carry on from the next cell.
%pip install -q -U "transformers>=5" "datasets>=4" accelerate scikit-learn

In [ ]:
import os, re, time, json, pathlib
import numpy as np
import pandas as pd
import torch

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

IN_COLAB = "google.colab" in str(globals().get("get_ipython", lambda: "")())

# Apple-silicon GPU (MPS) > CUDA > CPU. The Trainer picks this up on its own;
# printed here so the run log records what it trained on.
if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

ON_CUDA = device.type == "cuda"
USE_FP16 = ON_CUDA                 # mixed precision: big speedup on Colab, unsupported on MPS
NUM_WORKERS = 2 if ON_CUDA else 0  # MPS is happier single-threaded

print("torch:", torch.__version__, "| device:", device, "| colab:", IN_COLAB)
if ON_CUDA:
    print("gpu:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif IN_COLAB:
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then rerun.")
print("cwd:", os.getcwd())

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"
MAX_LENGTH = 128          # same length the Flask app tokenises with
NUM_LABELS = 3            # negative / neutral / positive

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label={0: "negative", 1: "neutral", 2: "positive"},
    label2id={"negative": 0, "neutral": 1, "positive": 2},
)
print(model.config.num_labels, "labels |", model.num_parameters() / 1e6, "M params")

### Smoke test on the pretrained model (before fine-tuning)

In [ ]:
tokens = tokenizer("This is great absolutely love this", return_tensors="pt")
with torch.no_grad():
    logits = model(**tokens).logits
print("logits:", logits.tolist())
print("predicted:", model.config.id2label[int(logits.argmax())])

## 2. Load the data

`twitter_training.csv` has **no header row**, so it must be read with `header=None` and
explicit names — otherwise the first tweet is eaten as the header and the columns end up
shifted. The four columns are: tweet id, entity, sentiment, tweet text.

In [ ]:
# Get the data into the session. Locally the files are already beside the notebook;
# on Colab pick whichever route you prefer.
TRAIN_CSV = "twitter_training.csv"
VAL_CSV = "archive/twitter_validation.csv"

if not os.path.exists(TRAIN_CSV):
    if IN_COLAB:
        # --- route A: Google Drive (put both CSVs in a folder and set DRIVE_DIR) ---
        # from google.colab import drive
        # drive.mount("/content/drive")
        # DRIVE_DIR = "/content/drive/MyDrive/sentiment-data"
        # TRAIN_CSV = f"{DRIVE_DIR}/twitter_training.csv"
        # VAL_CSV   = f"{DRIVE_DIR}/twitter_validation.csv"

        # --- route B: upload from your laptop (runs by default) ---
        from google.colab import files
        print("Upload twitter_training.csv (and twitter_validation.csv if you have it)...")
        up = files.upload()
        for name in up:
            if "validation" in name.lower():
                VAL_CSV = name
            elif "train" in name.lower():
                TRAIN_CSV = name
    else:
        raise FileNotFoundError(f"{TRAIN_CSV} not found next to the notebook")

print("train csv:", TRAIN_CSV, "| exists:", os.path.exists(TRAIN_CSV))
print("val csv:  ", VAL_CSV, "| exists:", os.path.exists(VAL_CSV))

In [ ]:
data = pd.read_csv(
    TRAIN_CSV,
    encoding="ISO-8859-1",
    header=None,
    names=["tweet_id", "entity", "sentiment", "text"],
)
print("rows:", len(data))
data.head()

In [ ]:
data = data[["text", "sentiment"]]

missing = int(data["text"].isna().sum())
data = data.dropna(subset=["text"]).reset_index(drop=True)
print(f"dropped {missing} rows with no tweet text; {len(data)} rows left")

print(data["sentiment"].value_counts())
data.head()

## 3. Slang / Gen-Z normalisation

Rewrites slang phrases into plain English before tokenising, so the model sees wording it
understands. `app/slang.py` holds the same table and `app/predictor.py` applies it the same
way at inference time.

In [ ]:
import re
slang_dict = {
    "party was lit": "party was amazing or exciting",
    "game is trash": "game is bad or disappointing",
    "lowkey not bad": "surprisingly decent",
    "clutch with the ride": "impressive or timely move, especially with the car",
    "movie is crap": "movie was terrible",
    "outfit is sick": "outfit looks really good",
    "acting salty": "being upset or bitter",
    "join you later": "will meet up soon",
    "vibe is chill": "atmosphere is relaxed",
    "that’s basic": "unoriginal or common",
    "album is fire": "album is excellent",
    "pulled through": "managed to succeed or show up",
    "deal was a W": "deal was a win or success",
    "fit is clean": "outfit looks sharp",
    "game is crack": "game is super addictive",
    "like a boss": "doing something confidently or impressively",
    "snack is gas": "snack is really good",
    "meme had me dead": "meme was hilarious",
    "skills are clutch": "skills are impressive in critical moments",
    "glow-up": "major improvement in appearance or skills",
    "squad pulled up": "group showed up",
    "top-tier suggestion": "really good suggestion",
    "obsessed with track": "really loves the song",
    "hoodie is drip": "hoodie is stylish",
    "kicks are fresh": "shoes look new and cool",
    "real one": "someone loyal and genuine",
    "comeback was savage": "comeback was brutal and impressive",
    "whole vibe": "overall mood is great",
    "understood assignment": "nailed the task perfectly",
    "big mood": "strongly relatable feeling",
    "straight up shade": "obvious insult or disrespect",
    "event was a flop": "event was a failure",
    "situation is sus": "situation seems suspicious",
    "that’s an L": "that’s a loss or failure",
    "excuse was weak": "poor excuse",
    "show is washed": "show is outdated or boring",
    "way too basic": "extremely unoriginal",
    "apology was mid": "apology was average, not convincing",
    "big yikes": "really awkward or bad",
    "attitude is dry": "unenthusiastic or boring attitude",
    "worst decision": "really bad choice",
    "plan is shaky": "plan is uncertain or unstable",
    "review is cap": "review is a lie or exaggeration",
    "wildin’": "acting crazy or out of control",
    "outfit is cheeks": "outfit looks bad",
    "project is falling off": "project is losing quality or momentum",
    "story was extra": "story was exaggerated or dramatic",
    "service is buns": "service was bad",
    "vibe is dry": "atmosphere is boring",
    "hot mess": "complete disaster",
    "see you there": "I’ll meet you at that place",
    "pull up later": "come by later",
    "vibe, just chillin’": "relaxed mood, just hanging out",
    "lowkey wanna stay in": "kind of want to stay home",
    "that’s facts": "totally true",
    "here for the vibes": "just enjoying the atmosphere",
    "plan, I’m down": "I’m okay with that plan",
    "kinda sus": "a bit suspicious",
    "see how it plays out": "wait and see what happens",
    "it is what it is": "accepting the situation",
    "safe move": "risk-free choice",
    "it’s aight": "it’s alright, not bad",
    "lowkey feelin’": "quietly liking something",
    "slide if I have time": "might come if free",
    "it’s chill": "it’s fine, no stress",
    "just lurking": "watching without engaging",
    "that’s a vibe": "that’s a good mood or feeling",
    "you do you": "do what makes you happy",
    "it’s whatever": "indifference or no strong opinion",
    "no biggie": "not a big deal",
    "song is straight fire": "song is really good",
    "GOAT for helping": "greatest of all time for helping",
    "game was lit": "game was exciting or awesome",
    "outfit was mad dope": "outfit looked amazing",
    "dessert is bussin’": "dessert tastes really good",
    "aced that exam": "did very well on the exam",
    "play was clutch": "critical and impressive move",
    "track slaps hard": "song is really good",
    "hoodie is sick": "hoodie looks great",
    "advice was on point": "advice was really good",
    "food is trash": "food is bad",
    "that’s cap": "that’s a lie",
    "update is dead": "update is boring or outdated",
    "movie was mid": "movie was average",
    "attitude is basic": "unoriginal attitude",
    "outfit is weak": "outfit doesn’t look good",
    "app is buggy": "app has many issues",
    "service is wack": "service was bad",
    "apology was fake": "insincere apology",
    "let you know later": "will inform you soon",
    "it was aight": "it was okay",
    "pulled up, we chillin’": "showed up and just hanging out",
    "link up tomorrow": "meet tomorrow",
    "slide through": "stop by",
    "party was mid": "party was average",
    "safe bet": "reliable choice",
    "just gonna dip": "leaving quickly",
    "party was a blast": "party was really fun",
    "performance off the chain": "performance was amazing",
    "nailed it": "did it perfectly",
    "legend for this": "really impressive for doing this",
    "vibe is immaculate": "atmosphere is perfect",
    "advice was clutch": "advice was crucial and helpful",
    "coffee hits different": "coffee tastes especially good today",
    "moves were fire": "moves were impressive",
    "service is trash": "service is bad",
    "excuse is weak": "poor excuse",
    "restaurant is lowkey mid": "restaurant is kind of average",
    "shady move": "suspicious action",
    "graphics are busted": "graphics look bad or are broken",
    "weather is bogus": "weather is disappointing",
    "kinda aight": "kind of okay"
}

def replace_slang(text):
    for slang, meaning in slang_dict.items():
        pattern = r'\b' + re.escape(slang) + r'\b'  # Ensures whole-word matching
        text = re.sub(pattern, meaning, text, flags=re.IGNORECASE)
    return text

In [ ]:
print(replace_slang("party was lit and the album is fire"))
print(replace_slang("service is trash, big yikes"))

In [ ]:
t0 = time.time()
data["clean_text"] = data["text"].astype(str).apply(replace_slang)
print(f"slang normalisation: {time.time() - t0:.1f}s")
data[["text", "clean_text"]].head()

## 4. Labels

The dataset carries a fourth class, `Irrelevant`, which is not a sentiment. The app is a
three-class model, so those rows are dropped rather than mapped — mapping them produced
`NaN` labels and made training impossible.

In [ ]:
data["sentiment"] = data["sentiment"].astype(str).str.strip().str.lower()
print("raw classes:", sorted(data["sentiment"].unique()))

sentiment_mapping = {"negative": 0, "neutral": 1, "positive": 2}
data["label"] = data["sentiment"].map(sentiment_mapping)

dropped = int(data["label"].isna().sum())
data = data.dropna(subset=["label"]).reset_index(drop=True)
data["label"] = data["label"].astype("int64")

print(f"dropped {dropped} rows (Irrelevant / unmapped); {len(data)} rows left")
print(data["label"].value_counts().sort_index())
assert data["label"].isna().sum() == 0
assert set(data["label"].unique()) == {0, 1, 2}

## 5. Tokenise

Batched call — one pass over the whole column instead of one tokenizer call per row.

In [ ]:
t0 = time.time()
encoded = tokenizer(
    data["clean_text"].tolist(),
    truncation=True,
    max_length=MAX_LENGTH,
    padding=False,          # padded per batch at training time (same as the app does)
)
print(f"tokenised {len(data)} rows in {time.time() - t0:.1f}s")

lengths = np.array([len(x) for x in encoded["input_ids"]])
print(f"token length: mean {lengths.mean():.1f}, median {np.median(lengths):.0f}, "
      f"p95 {np.percentile(lengths, 95):.0f}, max {lengths.max()}")

In [ ]:
labels = torch.tensor(data["label"].values, dtype=torch.long)

# Padded copy kept on disk for anything that wants fixed-width tensors.
padded = tokenizer(
    data["clean_text"].tolist(),
    truncation=True,
    max_length=MAX_LENGTH,
    padding="max_length",
    return_tensors="pt",
)
input_ids, attention_mask = padded["input_ids"], padded["attention_mask"]
torch.save((input_ids, attention_mask, labels), "final_dataset.pt")
print("saved final_dataset.pt", input_ids.shape, attention_mask.shape, labels.shape)

In [ ]:
# Reload and verify
input_ids, attention_mask, labels = torch.load("final_dataset.pt", weights_only=True)
print("Input IDs Shape:", input_ids.shape)
print("Attention Mask Shape:", attention_mask.shape)
print("Labels Shape:", labels.shape)
print("Unique Labels:", sorted(set(labels.tolist())))
print("NaNs in labels:", int(torch.isnan(labels.float()).sum()))

## 6. Train / validation split

In [ ]:
from datasets import Dataset

dataset = Dataset.from_dict({
    "input_ids": encoded["input_ids"],
    "attention_mask": encoded["attention_mask"],
    "labels": data["label"].tolist(),
})

split = dataset.train_test_split(test_size=0.2, seed=SEED, stratify_by_column=None)
train_dataset, val_dataset = split["train"], split["test"]
print("train:", len(train_dataset), "| val:", len(val_dataset))
print({k: type(v).__name__ for k, v in train_dataset[0].items()})

## 7. Training setup

Batch 32 with per-batch dynamic padding **and length-grouped batching**, which is where
almost all the speed comes from. Token lengths here are median 27, p99 90, max 128:

| batching | padded tokens | waste | measured |
|---|---|---|---|
| fixed `padding="max_length"` (128) | 7.8M | 4.16x | — |
| dynamic, random order | 5.3M | 2.82x | 50.6 samples/s |
| dynamic, length-grouped | 2.1M | 1.11x | **73.6 samples/s** |

Grouping rows of similar length into the same batch means batches pad to ~their own length
instead of to the longest tweet that happened to land in them. It changes nothing about the
result — only how much padding the GPU multiplies by zero.

`group_by_length` was removed from `TrainingArguments` in transformers 5, so the sampler is
wired in by hand below.

In [ ]:
from transformers import TrainingArguments, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

EPOCHS = 6
TRAIN_BS = 32
steps_per_epoch = max(1, len(train_dataset) // TRAIN_BS)
total_steps = steps_per_epoch * EPOCHS

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=TRAIN_BS,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    warmup_steps=int(0.06 * total_steps),   # warmup_ratio was removed in transformers 5
    weight_decay=0.01,
    eval_strategy="epoch",                  # renamed from evaluation_strategy
    save_strategy="epoch",
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=USE_FP16,                          # CUDA only; MPS has no fp16 autocast
    dataloader_num_workers=NUM_WORKERS,
    report_to="none",
    seed=SEED,
)
print(f"{steps_per_epoch} steps/epoch, {total_steps} total, warmup {training_args.warmup_steps}")
print(f"fp16: {USE_FP16} | workers: {NUM_WORKERS}")

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1,
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),
        "precision_macro": precision,
        "recall_macro": recall,
    }

In [ ]:
from transformers import Trainer, EarlyStoppingCallback
from transformers.trainer_pt_utils import LengthGroupedSampler


class LengthGroupedTrainer(Trainer):
    """Trainer that batches rows of similar length together.

    Replaces the `group_by_length=True` argument that transformers 5 removed. Measured
    1.46x faster than random-order batching on this dataset, with identical maths.
    """

    def _get_train_sampler(self, train_dataset=None):
        ds = train_dataset if train_dataset is not None else self.train_dataset
        if ds is None:
            return None
        return LengthGroupedSampler(
            batch_size=self.args.train_batch_size,
            dataset=ds,
            model_input_name="input_ids",
        )


trainer = LengthGroupedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,   # replaces the removed `tokenizer=` argument
    compute_metrics=compute_metrics,
    # 6 epochs will almost certainly overfit this dataset. Early stopping ends the run
    # once eval_loss has not improved for 2 epochs; load_best_model_at_end then restores
    # the best checkpoint, so the saved weights are the best ones either way.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
print("device the Trainer will use:", trainer.args.device)

## 8. Train

In [ ]:
t0 = time.time()
train_result = trainer.train()
print(f"\ntraining took {(time.time() - t0) / 60:.1f} min")
print(train_result.metrics)

## 9. Evaluate

In [ ]:
metrics = trainer.evaluate()
for k, v in metrics.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pred_out = trainer.predict(val_dataset)
y_pred = np.argmax(pred_out.predictions, axis=-1)
y_true = np.asarray(pred_out.label_ids)

print(classification_report(y_true, y_pred, target_names=["negative", "neutral", "positive"], digits=4))
print("confusion matrix (rows = true, cols = predicted)")
print(confusion_matrix(y_true, y_pred))

## 10. Save the weights

Written to `app/model/`, the directory `app/predictor.py` loads from. The sidebar in the
Flask app shows *“Your model is live”* once these files are present.

In [ ]:
MODEL_DIR = pathlib.Path("app/model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(MODEL_DIR))
tokenizer.save_pretrained(str(MODEL_DIR))

(MODEL_DIR / "training_metrics.json").write_text(json.dumps({
    "base_model": MODEL_NAME,
    "epochs": EPOCHS,
    "train_batch_size": TRAIN_BS,
    "max_length": MAX_LENGTH,
    "train_rows": len(train_dataset),
    "val_rows": len(val_dataset),
    "device": str(trainer.args.device),
    "eval_metrics": {k: (float(v) if isinstance(v, (int, float)) else v) for k, v in metrics.items()},
}, indent=2))

for p in sorted(MODEL_DIR.iterdir()):
    print(f"{p.name:30s} {p.stat().st_size / 1e6:8.2f} MB")

## 11. Check the saved model on the held-out validation file

`archive/twitter_validation.csv` is a separate file the model never trained on. Same shape
as the training file and no header, so it is read the same way.

In [ ]:
from transformers import AutoModelForSequenceClassification as _AutoCls, AutoTokenizer as _AutoTok

reloaded_tok = _AutoTok.from_pretrained(str(MODEL_DIR))
reloaded = _AutoCls.from_pretrained(str(MODEL_DIR)).to(device).eval()

HAVE_VAL = os.path.exists(VAL_CSV)
if HAVE_VAL:
    test_set = pd.read_csv(
        VAL_CSV,
        encoding="ISO-8859-1",
        header=None,
        names=["tweet_id", "entity", "sentiment", "text"],
    )
    print("rows:", len(test_set))
else:
    print(f"{VAL_CSV} not in this session - skipping section 11")
test_set.head() if HAVE_VAL else None

In [ ]:
label_map = {0: "Negative", 1: "Neutral", 2: "Positive"}

if HAVE_VAL:
    texts = [replace_slang(str(t)) for t in test_set["text"].tolist()]

    preds = []
    with torch.no_grad():
        for i in range(0, len(texts), 64):
            enc = reloaded_tok(texts[i:i + 64], return_tensors="pt", padding=True,
                               truncation=True, max_length=MAX_LENGTH).to(device)
            preds.extend(reloaded(**enc).logits.argmax(dim=-1).cpu().tolist())

    test_set["Predicted Sentiment"] = [label_map[p] for p in preds]
    display(test_set[["text", "sentiment", "Predicted Sentiment"]].head(10))

In [ ]:
# Accuracy on the three sentiment classes of the validation file (Irrelevant rows excluded,
# as they are not one of the model's outputs).
if HAVE_VAL:
    scored = test_set[test_set["sentiment"].str.strip().str.lower().isin(sentiment_mapping)].copy()
    y_true_v = scored["sentiment"].str.strip().str.lower().map(sentiment_mapping).to_numpy()
    y_pred_v = np.array([{"Negative": 0, "Neutral": 1, "Positive": 2}[p]
                         for p in scored["Predicted Sentiment"]])

    print(f"scored {len(scored)} of {len(test_set)} rows")
    print(classification_report(y_true_v, y_pred_v,
                                target_names=["negative", "neutral", "positive"], digits=4))
    print("\nFor reference, the untuned base model scores 0.6546 accuracy on this same file.")

## 12. Get the weights off Colab

`app/model/` lives in the Colab session and disappears when the runtime recycles, so pull it
down (or copy it to Drive) before you close the tab.

In [ ]:
import shutil

ZIP_BASE = "sentiment_model"
shutil.make_archive(ZIP_BASE, "zip", root_dir=str(MODEL_DIR))
zip_path = f"{ZIP_BASE}.zip"
print(f"{zip_path}: {os.path.getsize(zip_path) / 1e6:.1f} MB")

if IN_COLAB:
    # --- download to your laptop ---
    from google.colab import files
    files.download(zip_path)

    # --- or copy to Drive instead (mount it in section 2 first) ---
    # shutil.copy(zip_path, "/content/drive/MyDrive/sentiment_model.zip")

### Then, on your laptop

Unzip it into `app/model/` and start the app:

```bash
cd SentimentAnalyser
unzip -o ~/Downloads/sentiment_model.zip -d app/model
cd app && python app.py        # http://127.0.0.1:5000
```

The sidebar shows **"Your model is live"** once it loads. To read from somewhere else:
`SENTIMENT_MODEL_DIR=/path/to/model python app.py`

Note: `app/predictor.py` picks `cuda` or `cpu`, so on an Apple-silicon Mac it serves on CPU.
That is fine for one-message-at-a-time use — change the line to prefer `mps` if you want
batch CSV runs to go faster.